# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muneebulhaq02/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content page for one reporting date for one client in the daily performance table. For this assignment I use a mid-panel month (March 2026) because the internship guidelines recommend avoiding the final month when exploring features. For exploratory analysis, I use March 2026 because it is a complete mid-panel month and avoids using the final month where future outcomes may overlap.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import os
from google.colab import userdata

# Get token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Make it available to DuckDB
os.environ["HF_TOKEN"] = HF_TOKEN

con = duckdb.connect()

# Create Hugging Face secret
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

con.sql("""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┐
│  rows   │ start_date │  end_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features
- impressions
- clicks
- ctr
- gsc_avg_position
- sessions
- engagement_rate
- users
- new_users

## Context
- report_date
- client_hash_id
- content_hash_id

## Label / Proxy

For this exploratory stage, no observed label exists in the warehouse. A proxy label will be defined later using future performance trends for the ranking task.

## Excluded fields

- future outcome information
- columns created from future months
- identifiers used only for joins (not model features)

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = [
    "impressions",
    "clicks",
    "gsc_avg_position",
    "sessions",
    "ctr"
]

context = [
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

excluded = [
    "future information",
    "label-derived columns",
    "product decision fields"
]

print("Features:")
print(features)

print("\nContext:")
print(context)

print("\nExcluded:")
print(excluded)


Features:
['impressions', 'clicks', 'gsc_avg_position', 'sessions', 'ctr']

Context:
['client_hash_id', 'content_hash_id', 'report_date']

Excluded:
['future information', 'label-derived columns', 'product decision fields']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The following queries verify the data contract.

1. Confirm the grain by checking for duplicate rows.
2. Count rows and verify the date window.
3. Check availability of GA4 data using `ga4_data_available IS TRUE`.

In [4]:
import duckdb
import os
from google.colab import userdata

# Get token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Make it available to DuckDB
os.environ["HF_TOKEN"] = HF_TOKEN

con = duckdb.connect()

# Create Hugging Face secret
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Duplicate grain check

con.sql("""
SELECT
COUNT(*) total_rows,
COUNT(gsc_avg_position) position_present,
COUNT(sessions_organic) sessions_present
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

print("\n--------------------------------\n")

con.sql("""
SELECT
report_date,
client_hash_id,
content_hash_id,
COUNT(*) AS duplicates
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
report_date,
client_hash_id,
content_hash_id
HAVING COUNT(*)>1
LIMIT 10
""").show()

print("\n--------------------------------\n")

# Row count + date window

con.sql("""
SELECT
COUNT(*) AS rows,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").show()

print("\n--------------------------------\n")

# GA4 availability

con.sql("""
SELECT
COUNT(*) AS rows_with_ga4
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS TRUE
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────┬──────────────────┐
│ total_rows │ position_present │ sessions_present │
│   int64    │      int64       │      int64       │
├────────────┼──────────────────┼──────────────────┤
│    9841378 │          3611061 │          6822637 │
└────────────┴──────────────────┴──────────────────┘


--------------------------------



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬────────────┐
│ report_date │ client_hash_id │ content_hash_id │ duplicates │
│    date     │    varchar     │     varchar     │   int64    │
├─────────────┴────────────────┴─────────────────┴────────────┤
│                           0 rows                            │
└─────────────────────────────────────────────────────────────┘


--------------------------------

┌─────────┬────────────┬────────────┐
│  rows   │ start_date │  end_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘


--------------------------------



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┐
│ rows_with_ga4 │
│     int64     │
├───────────────┤
│        413966 │
└───────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset supports decision-support analysis rather than causal conclusions.

Some important limitations are:

- Different clients have different amounts of historical data (an unbalanced panel).
- Some early rows contain only Google Search Console data because GA4 tracking had not yet started.
- The final month of the dataset should not be used when developing labels because it naturally becomes the future outcome window.
- This analysis can identify useful patterns for prioritizing content review, but it cannot prove that refreshing a page will cause better search performance or predict Google's ranking algorithm.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
limitations = [
    "Unbalanced client history",
    "Some rows contain only GSC data",
    "Final month reserved for testing",
    "Cannot prove causation",
    "Cannot predict Google's algorithm"
]

print("Data Limitations\n")

for item in limitations:
    print("-", item)

Data Limitations

- Unbalanced client history
- Some rows contain only GSC data
- Final month reserved for testing
- Cannot prove causation
- Cannot predict Google's algorithm


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.